# Thulla DMC — Colab training (CPU or T4)

Minimal DouZero-style self-play for a decent 4-player Thulla bot.

Learns **card plays** and **ASK/PASS** on the take phase (victims always give).

Obs include known holdings (thulla/take) and the **free unknown** card pool; heads-up uses deduced opponent hands.

**Default below:** CPU learner + 6 CPU actors (works without a GPU). For T4, set `--training_device 0` and add `--require_gpu`.

- Latest checkpoint every ~3 min → `model.tar`
- **Eval vs random** every 15 min / **heuristic** every 30 min → `model_best.tar` on improvement
- Fixed `--eval_seed` (default 10000) for comparable evals across checkpoints
- Eval summaries → **`eval_log.csv`** on Drive

**Note:** If you changed obs/encoding recently, start a **fresh** checkpoint folder or delete old `model.tar`.


## 1. Install deps

In [ ]:
%pip install -q "numpy>=1.24" "torch>=2.0"

## 2. Get thulla-ai code

Either clone your repo, or upload a zip of `thulla-ai` to Drive and set `REPO_DIR` below.

In [ ]:
import os
import sys

import torch

# --- edit these ---
REPO_URL = ""  # e.g. "https://github.com/YOU/thulla-ai.git" or leave blank if uploading
REPO_DIR = "/content/thulla-ai"
DRIVE_CKPT = "/content/drive/MyDrive/thulla_dmc_ckpts"
# ------------------

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_CKPT, exist_ok=True)

if REPO_URL and not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
elif not os.path.isdir(REPO_DIR):
    raise SystemExit(
        f"Missing {REPO_DIR}. Set REPO_URL or upload thulla-ai there (must contain thulla/ and thulla_dmc/)."
    )

sys.path.insert(0, REPO_DIR)

print("REPO_DIR:", REPO_DIR)
print("checkpoints:", DRIVE_CKPT)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("CUDA: not available (CPU training OK — use --training_device cpu, no --require_gpu)")


## 3. Train (auto-resume if checkpoint exists)

**CPU preset** below: 3 actors × 8 envs/actor (batched forward), Torch threads=1. Timed evals: random every 15 min, heuristic every 30 min → `model_best.tar`.

Fixed `--eval_seed 10000`: same deals across checkpoints.

For T4: `--training_device 0`, `--num_actors 6`, add `--require_gpu`.


In [ ]:
import os
from thulla_dmc.arguments import parse_args
from thulla_dmc.train import train

ckpt_dir = os.path.join(DRIVE_CKPT, "thulla_dmc")
has_ckpt = os.path.exists(os.path.join(ckpt_dir, "model.tar"))

# Colab CPU preset — for T4: use "--training_device", "0" and add "--require_gpu"
argv = [
    "--savedir", DRIVE_CKPT,
    "--xpid", "thulla_dmc",
    "--training_device", "cpu",
    "--num_actors", "3",
    "--actor_envs", "8",
    "--batch_size", "512",
    "--save_interval", "3",
    "--eval_random_minutes", "15",
    "--eval_heuristic_minutes", "30",
    "--eval_games", "50",
    "--eval_seed", "10000",
    "--total_episodes", "350000",
    "--exp_epsilon", "0.05",
    "--log_interval", "25",
]
if has_ckpt:
    argv.append("--load_model")
    print("Resuming from", ckpt_dir)
else:
    print("Starting fresh →", ckpt_dir)

flags = parse_args(argv)
train(flags)


## 4. Manual evaluate

Use `model_best.tar` if present (best vs heuristic); otherwise `model.tar`.
Primary metric: **P(not last)**.


In [ ]:
import os
from thulla_dmc.evaluate import evaluate

best = f"{DRIVE_CKPT}/thulla_dmc/model_best.tar"
latest = f"{DRIVE_CKPT}/thulla_dmc/model.tar"
ckpt = best if os.path.exists(best) else latest
print("Evaluating:", ckpt)

print("\n--- vs random ---")
evaluate(ckpt, num_games=200, device="cpu", opponent="random")

print("\n--- vs heuristic ---")
evaluate(ckpt, num_games=100, device="cpu", opponent="heuristic")